# ⚡ TalkForge — AI Talking-Head Video Generator

Upload a portrait + audio → get a lip-synced talking video.

### How to use
1. **Runtime → Change runtime type → T4 GPU** (do this first!)
2. **Runtime → Run all**
3. Wait ~8 minutes on first run
4. Click the `gradio.live` public URL that prints at the bottom

> ⚠️ The notebook auto-restarts the kernel once after installing packages. That is expected — Colab will continue running the remaining cells automatically.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — GPU check
# ═══════════════════════════════════════════════════════════
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('\n'.join(r.stdout.split('\n')[:12]))
    print('\n✅ GPU detected — ready to proceed.')
else:
    print('❌  No GPU found.')
    print('   → Runtime → Change runtime type → T4 GPU → Save')
    print('   → Then Runtime → Disconnect and delete runtime → Run all')
    sys.exit(1)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — System packages (FFmpeg, cmake, build tools)
# ═══════════════════════════════════════════════════════════
print('Installing system packages…')
import subprocess
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run([
    'apt-get', 'install', '-y', '-qq',
    'ffmpeg', 'cmake', 'build-essential',
    'libboost-all-dev', 'libopenblas-dev',
    'liblapack-dev', 'libatlas-base-dev',
    'wget', 'curl', 'git', 'unzip'
], check=True)

r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('FFmpeg:', r.stdout.split('\n')[0])
print('✅ System packages ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Clone repos
# ═══════════════════════════════════════════════════════════
import os, subprocess

TF_DIR = '/content/TalkForge'
ST_DIR = '/content/TalkForge/SadTalker'

if not os.path.exists(os.path.join(TF_DIR, 'main.py')):
    print('Cloning TalkForge…')
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/rydenxgod-bot/TalkForge.git', TF_DIR], check=True)
else:
    print('✓ TalkForge already present')

if not os.path.exists(os.path.join(ST_DIR, 'inference.py')):
    print('Cloning SadTalker…')
    subprocess.run(['git', 'clone', '--depth', '1',
        'https://github.com/OpenTalker/SadTalker.git', ST_DIR], check=True)
else:
    print('✓ SadTalker already present')

print('✅ Repos ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Install Python packages
#
# KEY STRATEGY:
#   • Do NOT install SadTalker requirements.txt — it pins numpy<2
#     which breaks Colab's pre-built pandas/gradio C extensions.
#   • Keep numpy at 2.x (Colab default). Instead we patch the one
#     deprecated numpy call SadTalker makes at runtime.
#   • After install, kernel MUST restart so Python reloads the
#     freshly installed native extensions cleanly.
# ═══════════════════════════════════════════════════════════
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + list(args), check=True)

print('[1/6] Core ML stack…')
pip('torch', 'torchvision', 'torchaudio', '--index-url', 'https://download.pytorch.org/whl/cu121')

print('[2/6] Gradio + numpy (keep numpy 2.x)…')
pip('gradio==4.44.1', 'numpy>=2.0')

print('[3/6] Audio/vision…')
pip(
    'librosa', 'soundfile', 'pillow',
    'opencv-python-headless', 'imageio', 'imageio-ffmpeg',
    'scipy', 'tqdm', 'pyyaml'
)

print('[4/6] Face analysis…')
pip('face_alignment', 'facexlib', 'gfpgan', 'realesrgan', 'basicsr')

print('[5/6] Model utilities…')
pip('safetensors', 'kornia', 'einops', 'yacs', 'batch-face', 'mediapipe')

print('[6/6] dlib (may take a few minutes to compile)…')
pip('dlib')

print()
print('✅ All packages installed.')
print()
print('━' * 55)
print('  🔄  Restarting kernel to flush binary incompatibilities…')
print('  Colab will automatically continue with Cell 5.')
print('━' * 55)

# Auto-restart so freshly compiled C extensions load cleanly
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — Patch SadTalker for numpy 2.x compatibility
#
# SadTalker uses np.bool, np.int, np.float (removed in numpy 2).
# We patch the affected files in-place so no source edits are needed.
# ═══════════════════════════════════════════════════════════
import os, re

ST_DIR = '/content/TalkForge/SadTalker'

NUMPY_ALIASES = [
    # (old, new)
    (r'\bnp\.bool\b',    'np.bool_'),
    (r'\bnp\.int\b',     'np.int_'),
    (r'\bnp\.float\b',   'np.float64'),
    (r'\bnp\.complex\b', 'np.complex128'),
    (r'\bnp\.object\b',  'np.object_'),
    (r'\bnp\.str\b',     'np.str_'),
]

patched = []
for root, _, files in os.walk(ST_DIR):
    for fname in files:
        if not fname.endswith('.py'):
            continue
        fpath = os.path.join(root, fname)
        try:
            with open(fpath, 'r', encoding='utf-8', errors='ignore') as f:
                src = f.read()
            new_src = src
            for old, new in NUMPY_ALIASES:
                new_src = re.sub(old, new, new_src)
            if new_src != src:
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(new_src)
                patched.append(os.path.relpath(fpath, ST_DIR))
        except Exception:
            pass

if patched:
    print(f'Patched {len(patched)} file(s) for numpy 2.x:')
    for p in patched:
        print(f'  • {p}')
else:
    print('No numpy compat patches needed.')

# Also patch basicsr which has the same issue
import subprocess, sys
import pkg_resources
try:
    basicsr_path = pkg_resources.get_distribution('basicsr').location
    bsr_dir = os.path.join(basicsr_path, 'basicsr')
    bsr_patched = []
    for root, _, files in os.walk(bsr_dir):
        for fname in files:
            if not fname.endswith('.py'):
                continue
            fpath = os.path.join(root, fname)
            try:
                with open(fpath, 'r', errors='ignore') as f:
                    src = f.read()
                new_src = src
                for old, new in NUMPY_ALIASES:
                    new_src = re.sub(old, new, new_src)
                if new_src != src:
                    with open(fpath, 'w') as f:
                        f.write(new_src)
                    bsr_patched.append(fname)
            except Exception:
                pass
    if bsr_patched:
        print(f'Patched {len(bsr_patched)} basicsr file(s) for numpy 2.x')
except Exception as e:
    print(f'basicsr patch skipped: {e}')

print('✅ Numpy 2.x patches applied.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — Download SadTalker model weights
# ═══════════════════════════════════════════════════════════
import os, subprocess

TF_DIR   = '/content/TalkForge'
ST_DIR   = '/content/TalkForge/SadTalker'
W_DIR    = '/content/TalkForge/weights'
CKPT_DIR = os.path.join(ST_DIR, 'checkpoints')

os.makedirs(W_DIR,    exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

HF = 'https://huggingface.co/vinthony/SadTalker/resolve/main'

FILES = [
    ('SadTalker_V0.0.2_256.safetensors', f'{HF}/SadTalker_V0.0.2_256.safetensors'),
    ('SadTalker_V0.0.2_512.safetensors', f'{HF}/SadTalker_V0.0.2_512.safetensors'),
    ('mapping_00109-model.pth.tar',       f'{HF}/mapping_00109-model.pth.tar'),
    ('mapping_00229-model.pth.tar',       f'{HF}/mapping_00229-model.pth.tar'),
    ('epoch_20.pth',                      f'{HF}/epoch_20.pth'),
]

def wget(url, dest):
    subprocess.run(['wget', '-q', '--show-progress', '-O', dest, url], check=True)

for fname, url in FILES:
    dest = os.path.join(W_DIR, fname)
    link = os.path.join(CKPT_DIR, fname)
    if not os.path.exists(dest):
        print(f'↓ {fname}')
        wget(url, dest)
    else:
        print(f'✓ {fname}')
    if not os.path.exists(link):
        os.symlink(dest, link)

# BFM face model
BFM_ZIP  = os.path.join(W_DIR, 'BFM_Fitting.zip')
BFM_DIR  = os.path.join(W_DIR, 'BFM_Fitting')
BFM_LINK = os.path.join(CKPT_DIR, 'BFM_Fitting')

if not os.path.exists(BFM_DIR):
    print('↓ BFM_Fitting.zip')
    wget(f'{HF}/BFM_Fitting.zip', BFM_ZIP)
    subprocess.run(['unzip', '-q', BFM_ZIP, '-d', W_DIR], check=True)
    print('✓ BFM extracted')
else:
    print('✓ BFM_Fitting')

if not os.path.exists(BFM_LINK):
    os.symlink(BFM_DIR, BFM_LINK)

print('\n✅ SadTalker weights ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Download face enhancer (GFPGAN) weights
# ═══════════════════════════════════════════════════════════
import os, subprocess, shutil

ST_DIR = '/content/TalkForge/SadTalker'
GFP_DIR = os.path.join(ST_DIR, 'gfpgan', 'weights')
os.makedirs(GFP_DIR, exist_ok=True)

WEIGHTS = [
    ('GFPGANv1.4.pth',
     'https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth'),
    ('detection_Resnet50_Final.pth',
     'https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth'),
    ('parsing_parsenet.pth',
     'https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth'),
]

def wget(url, dest):
    subprocess.run(['wget', '-q', '--show-progress', '-O', dest, url], check=True)

for fname, url in WEIGHTS:
    dest = os.path.join(GFP_DIR, fname)
    if not os.path.exists(dest):
        print(f'↓ {fname}')
        wget(url, dest)
    else:
        print(f'✓ {fname}')

# Copy to facexlib's default cache dir
FX_CACHE = os.path.expanduser('~/.cache/facexlib')
os.makedirs(FX_CACHE, exist_ok=True)
for fname in ('detection_Resnet50_Final.pth', 'parsing_parsenet.pth'):
    src  = os.path.join(GFP_DIR, fname)
    dst  = os.path.join(FX_CACHE, fname)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  copied {fname} → facexlib cache')

print('\n✅ GFPGAN weights ready.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Final smoke test
# ═══════════════════════════════════════════════════════════
import sys, os

TF_DIR = '/content/TalkForge'
ST_DIR = '/content/TalkForge/SadTalker'

for p in [TF_DIR, ST_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)
os.chdir(TF_DIR)

all_ok = True

IMPORT_TESTS = [
    ('numpy',         'import numpy as np; print(f"  numpy {np.__version__}")'),
    ('torch',         'import torch; print(f"  torch {torch.__version__}, CUDA={torch.cuda.is_available()}")'),
    ('gradio',        'import gradio; print(f"  gradio {gradio.__version__}")'),
    ('cv2',           'import cv2; print(f"  cv2 {cv2.__version__}")'),
    ('librosa',       'import librosa; print(f"  librosa {librosa.__version__}")'),
    ('face_alignment','import face_alignment; print("  face_alignment OK")'),
    ('gfpgan',        'import gfpgan; print("  gfpgan OK")'),
    ('basicsr',       'import basicsr; print("  basicsr OK")'),
    ('safetensors',   'import safetensors; print("  safetensors OK")'),
]

print('Import checks:')
for name, stmt in IMPORT_TESTS:
    try:
        exec(stmt)
    except Exception as e:
        print(f'  ✗ {name}: {e}')
        all_ok = False

print('\nWeight checks:')
REQUIRED = [
    '/content/TalkForge/SadTalker/checkpoints/SadTalker_V0.0.2_256.safetensors',
    '/content/TalkForge/SadTalker/checkpoints/epoch_20.pth',
    '/content/TalkForge/SadTalker/gfpgan/weights/GFPGANv1.4.pth',
    '/content/TalkForge/SadTalker/gfpgan/weights/detection_Resnet50_Final.pth',
    '/content/TalkForge/SadTalker/gfpgan/weights/parsing_parsenet.pth',
]
for f in REQUIRED:
    exists = os.path.exists(f)
    print(f'  {"✓" if exists else "✗"} {os.path.basename(f)}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('✅ All checks passed — ready to launch!')
else:
    print('⚠️  Some checks failed.')
    print('   If imports failed: Runtime → Restart session, then re-run from Cell 5.')
    print('   If weights are missing: re-run Cells 6 and 7.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — Launch TalkForge 🚀
# ═══════════════════════════════════════════════════════════
import sys, os

TF_DIR = '/content/TalkForge'
ST_DIR = '/content/TalkForge/SadTalker'

for p in [TF_DIR, ST_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(TF_DIR)
os.makedirs(os.path.join(TF_DIR, 'outputs'), exist_ok=True)

# Silence noisy warnings that clutter Colab output
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS'] = 'ignore'

print('═' * 60)
print('  ⚡  TalkForge — starting Gradio…')
print('  Public URL appears below in ~20 seconds.')
print('  Share it with anyone — valid for 72 hours.')
print('═' * 60)

from app.ui import launch
launch(share=True, port=7860)